# 3h30 au départ de Paris : analyse comparée de cinq modes de transport

*Petite géographie d'une matinée passée à attendre mon tour.*

Ce notebook contient l'ensemble de la chaîne de traitement du projet : à partir de données ouvertes, il détermine
ce qui est atteignable depuis l'Île-de-France en **3h30 passées à bord**, à pied, à vélo, en voiture, en train et en avion.

La méthode, les choix et les limites sont détaillés dans [`docs/methode.md`](../docs/methode.md).
Les sources à télécharger avant l'exécution sont décrites dans [`data/README.md`](../data/README.md).

| Paramètre | Valeur retenue |
|---|---|
| Budget-temps | 210 minutes de temps à bord, identique pour tous les modes |
| Jour de référence | mardi 6 octobre 2026, meilleur trajet de la journée |
| Marche, vélo, voiture | départ de la gare du Nord ; trafic fluide pour la voiture |
| Train | 7 terminus parisiens + gares TGV de Roissy-CDG 2, Marne-la-Vallée–Chessy et du pôle de Massy ; trajets directs |
| Avion | vols directs au départ de Roissy-CDG et d'Orly |
| Destinations | hors Île-de-France (sauf pour la marche et le vélo, représentés par leur zone) |

**Sommaire**
1. Train : horaires GTFS SNCF
2. Voiture : grille et matrice de temps openrouteservice
3. Avion : liaisons directes et modèle de durée
4. Marche et vélo : isochrones openrouteservice
5. Synthèse : comparaison des modes et communes atteintes à vélo
6. Préparation des couches pour la cartographie

## 0. Configuration

Le notebook s'exécute depuis la racine du dépôt (`jupyter lab` lancé dans ce dossier, puis ouvrir `notebooks/traitement.ipynb`).
Tous les chemins sont relatifs à la racine grâce à la cellule ci-dessous.

In [ ]:
import os
import re
import time
import unicodedata
from io import StringIO
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import requests
from pyproj import Geod
from shapely.geometry import Point

# Se placer à la racine du dépôt, que le notebook soit lancé depuis la racine ou depuis notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

DATA = Path("data")
GTFS = DATA / "gtfs_sncf"
ADMIN_EXPRESS = DATA / "ADMIN-EXPRESS-COG.gpkg"
PAYS_GISCO = DATA / "CNTR_RG_03M_2020_3035.geojson"
AIRPORTS = DATA / "airports.csv"
OUT = Path("resultats")
OUT.mkdir(exist_ok=True)
GPKG = OUT / "resultats.gpkg"

BUDGET_MIN = 210          # budget-temps à bord, en minutes (3h30)
JOUR = "20261006"         # jour de référence (format GTFS)
ORIG = [2.3553, 48.8809]  # gare du Nord [longitude, latitude], départ des modes routiers
SEUILS = [60, 120, 180, BUDGET_MIN]

GARES_ORIGINE = [
    "StopArea:OCE87271007",  # Paris Gare du Nord
    "StopArea:OCE87113001",  # Paris Est
    "StopArea:OCE87384008",  # Paris Saint-Lazare
    "StopArea:OCE87391003",  # Paris Montparnasse Hall 1-2
    "StopArea:OCE87391102",  # Paris Montparnasse Vaugirard
    "StopArea:OCE87547000",  # Paris Austerlitz
    "StopArea:OCE87686006",  # Paris Gare de Lyon
    "StopArea:OCE87686667",  # Paris Bercy
    "StopArea:OCE87271494",  # Aéroport CDG 2 TGV
    "StopArea:OCE87111849",  # Marne-la-Vallée Chessy
    "StopArea:OCE87393702",  # Massy TGV
    "StopArea:OCE87393579",  # Massy-Palaiseau (pôle de Massy)
]

# 1. Train

## 1.1 Inventaire du GTFS SNCF

Le jeu « Horaires SNCF » couvre les TGV, Intercités et TER opérés par SNCF Voyageurs.
On recherche d'abord les gares parisiennes et internationales candidates, les opérateurs présents et la période couverte.

In [ ]:
stops = pd.read_csv(GTFS / "stops.txt", dtype=str)
motifs = "Paris|Chessy|Massy|Roissy|London|Londres|Bruxelles|Amsterdam"
cand = stops[stops["stop_name"].str.contains(motifs, case=False, na=False)]
print(cand[["stop_id", "stop_name", "location_type", "parent_station"]].to_string())

                                       stop_id                                   stop_name location_type        parent_station
328                       StopArea:OCE87111849                      Marne-la-Vallée Chessy             1                   NaN
329                StopPoint:OCEOUIGO-87111849                      Marne-la-Vallée Chessy             0  StopArea:OCE87111849
330            StopPoint:OCETGV INOUI-87111849                      Marne-la-Vallée Chessy             0  StopArea:OCE87111849
341                       StopArea:OCE87113001                                   Paris Est             1                   NaN
342              StopPoint:OCECar TER-87113001                                   Paris Est             0  StopArea:OCE87113001
343                  StopPoint:OCEICE-87113001                                   Paris Est             0  StopArea:OCE87113001
344                StopPoint:OCEOUIGO-87113001                                   Paris Est             0  StopA

La gare de Roissy porte le nom « Aéroport Charles de Gaulle 2 TGV » : le filtre par nom ne la trouve pas.
C'est pourquoi l'origine est définie par une **liste d'identifiants** (`GARES_ORIGINE`) plutôt que par un motif.

In [ ]:
print(stops[stops["stop_name"].str.contains("Charles|Aéroport|Aeroport", case=False, na=False)][["stop_id", "stop_name", "location_type"]].to_string())

                                      stop_id                         stop_name location_type
210                      StopArea:OCE85010264                   Geneve-Aeroport             1
211           StopPoint:OCETrain TER-85010264                   Geneve-Aeroport             0
1542                     StopArea:OCE87214528                 Entzheim Aéroport             1
1543            StopPoint:OCECar TER-87214528                 Entzheim Aéroport             0
1544          StopPoint:OCETrain TER-87214528                 Entzheim Aéroport             0
1649                     StopArea:OCE87271494  Aéroport Charles de Gaulle 2 TGV             1
1650              StopPoint:OCEOUIGO-87271494  Aéroport Charles de Gaulle 2 TGV             0
1651          StopPoint:OCETGV INOUI-87271494  Aéroport Charles de Gaulle 2 TGV             0
3062                     StopArea:OCE87439554    Vitrolles Aéroport Marseille P             1
3063            StopPoint:OCECar TER-87439554    Vitrolles A

In [ ]:
print(pd.read_csv(GTFS / "agency.txt", dtype=str)["agency_name"].unique())
cal = pd.read_csv(GTFS / "calendar_dates.txt", dtype=str)
print(cal["date"].min(), cal["date"].max())

Sortie obtenue lors de l'exécution : opérateurs SNCF Voyageurs, CFF et TMR (Eurostar est absent du flux) ;
période couverte du 26/09/2026 au 28/02/2027. Le flux ne contient que `calendar_dates.txt` : les jours de circulation sont décrits date par date.

## 1.2 Trains circulant le jour de référence

On garde les circulations actives le 6 octobre 2026, hors lignes de bus (`route_type = 3`),
puis les passages par les gares d'origine.

In [ ]:
services = cal[(cal["date"] == JOUR) & (cal["exception_type"] == "1")]["service_id"]
trips = pd.read_csv(GTFS / "trips.txt", dtype=str)
routes = pd.read_csv(GTFS / "routes.txt", dtype=str)
trips_jour = trips[trips["service_id"].isin(services)].merge(routes[["route_id", "route_type"]], on="route_id")
trips_jour = trips_jour[trips_jour["route_type"] != "3"]

In [ ]:
st = pd.read_csv(GTFS / "stop_times.txt", dtype=str)
st = st[st["trip_id"].isin(trips_jour["trip_id"])]
st = st.merge(stops[["stop_id", "parent_station"]], on="stop_id")
trips_origine = st[st["parent_station"].isin(GARES_ORIGINE)]["trip_id"].unique()
print(len(trips_jour), "trains le", JOUR, "/", len(trips_origine), "passent par une gare d'origine")
prod = st[st["parent_station"].isin(GARES_ORIGINE)]["stop_id"].str.extract(r"OCE(.+)-\d+$")[0]
print(prod.value_counts())

10178 trains le 20261006 / 1258 passent par une gare d'origine
0
TGV INOUI             591
Train TER             568
OUIGO                  89
INTERCITES             36
Lyria                  28
ICE                    24
INTERCITES de nuit     10
Train                   4
Car TER                 2
Car à réservation       2
Name: count, dtype: int64


Des cars (« Car TER », « Car à réservation ») subsistent malgré le filtre sur `route_type`. Le diagnostic montre
que leurs lignes sont codées comme ferroviaires (`route_type = 2`) : certaines circulations de lignes de train sont
assurées par car (substitutions). Dans ce flux, **le mode se lit au niveau de la circulation**, dans le produit
contenu dans l'identifiant du `StopPoint`.

In [ ]:
cibles = st[st["parent_station"].isin(GARES_ORIGINE) & st["stop_id"].str.contains("OCECar|OCETrain-")]
ids_routes = trips_jour[trips_jour["trip_id"].isin(cibles["trip_id"])]["route_id"]
print(routes[routes["route_id"].isin(ids_routes)].to_string())
print(routes["route_type"].value_counts())

                                           route_id agency_id route_short_name         route_long_name route_desc route_type route_url route_color route_text_color
224  FR:Line::5648ced8-53ec-4327-a902-803b8aeeb1d9:      1187             770A        Paris - Briançon        NaN          2       NaN         NaN              NaN
392  FR:Line::9302e13e-8434-45ce-a5bc-9fe445fd95bc:      1187               K4        PARIS - MULHOUSE        NaN          2       NaN      BF005F           FFFFFF
409  FR:Line::97E911FA-C8F2-4861-AD3C-C36CF4750A49:      1187               C1  ROUEN - VERNON - PARIS        NaN          2       NaN      0749FF           FFFFFF
631  FR:Line::d1189a05-48e1-477d-9b3c-f8ac8cb0397b:      1187             361A       Paris - Bruxelles        NaN          2       NaN         NaN              NaN
route_type
2    604
3    117
0      4
Name: count, dtype: int64


In [ ]:
st["produit"] = st["stop_id"].str.extract(r"OCE(.+)-\d+$")[0]
est_car = st["produit"].str.startswith("Car", na=False)
trips_car = st.loc[est_car, "trip_id"].unique()
mixtes = st[st["trip_id"].isin(trips_car) & ~est_car]["trip_id"].unique()
print(len(trips_car), "circulations avec car /", len(mixtes), "mixtes train+car")
st = st[~st["trip_id"].isin(trips_car)]
trips_origine = st[st["parent_station"].isin(GARES_ORIGINE)]["trip_id"].unique()
print(len(trips_origine), "trains passant par une gare d'origine")
print(st[st["parent_station"].isin(GARES_ORIGINE)]["produit"].value_counts())

961 circulations avec car / 0 mixtes train+car
1254 trains passant par une gare d'origine
produit
TGV INOUI             591
Train TER             568
OUIGO                  89
INTERCITES             36
Lyria                  28
ICE                    24
INTERCITES de nuit     10
Train                   4
Name: count, dtype: int64


Aucune circulation mixte : l'exclusion des circulations en car ne retire aucun tronçon ferroviaire.
Le filtre par `route_type` seul aurait laissé passer 961 circulations en car à l'échelle nationale.

## 1.3 Durée du trajet direct le plus rapide vers chaque gare

Règles appliquées :
1. **Sens de circulation** : seuls les arrêts situés après la gare de montée sont retenus (`stop_sequence` converti en entier).
2. **Durée** : heure d'arrivée moins heure de départ, calculées en secondes pour gérer les horaires GTFS au-delà de minuit (`25:30:00`).
3. **Plusieurs gares d'origine sur un même train** : la montée donnant la durée la plus courte est retenue.
4. **Montée et descente autorisées** : `pickup_type` et `drop_off_type` différents de 1.
5. **Agrégation** : durée minimale de la journée par gare de destination, gares d'origine exclues.

In [ ]:
def en_sec(s):
    p = s.str.split(":", expand=True).astype(int)
    return p[0] * 3600 + p[1] * 60 + p[2]

st["seq"] = st["stop_sequence"].astype(int)
st["dep"] = en_sec(st["departure_time"])
st["arr"] = en_sec(st["arrival_time"])
for c in ["pickup_type", "drop_off_type"]:
    st[c] = st[c].fillna("0") if c in st else "0"

In [ ]:
o = st[st["parent_station"].isin(GARES_ORIGINE) & (st["pickup_type"] != "1")]
o = o[["trip_id", "seq", "dep", "parent_station", "produit"]]
d = st[~st["parent_station"].isin(GARES_ORIGINE) & (st["drop_off_type"] != "1")]
d = d[["trip_id", "seq", "arr", "parent_station"]]
od = o.merge(d, on="trip_id", suffixes=("_o", "_d"))
od = od[od["seq_d"] > od["seq_o"]]
od["duree_min"] = (od["arr"] - od["dep"]) / 60

In [ ]:
res = od.sort_values("duree_min").drop_duplicates("parent_station_d")
res = res[res["duree_min"] <= BUDGET_MIN]
res = res.merge(stops[["stop_id", "stop_name"]], left_on="parent_station_d", right_on="stop_id")
res = res.sort_values("duree_min")
cols = ["stop_name", "duree_min", "parent_station_o", "produit"]
print(len(res), f"gares atteignables en {BUDGET_MIN} min ou moins")
print(res[cols].head(15).to_string())
print(res[cols].tail(15).to_string())

334 gares atteignables en 3h30 ou moins
                          stop_name  duree_min      parent_station_o    produit
0              Versailles Chantiers       11.0  StopArea:OCE87391003  Train TER
1              Orry-la-Ville - Coye       18.0  StopArea:OCE87271007  Train TER
2         Saint-Quentin en Yvelines       20.0  StopArea:OCE87391003  Train TER
3   Dammartin - Juilly - Saint-Mard       23.0  StopArea:OCE87271007  Train TER
4              Chantilly - Gouvieux       23.0  StopArea:OCE87271007  Train TER
5                             Creil       25.0  StopArea:OCE87271007  Train TER
6                TGV Haute Picardie       26.0  StopArea:OCE87271494  TGV INOUI
7             Champagne-Ardenne TGV       29.0  StopArea:OCE87111849  TGV INOUI
8                 Persan - Beaumont       29.0  StopArea:OCE87271007  Train TER
9                       Rambouillet       29.0  StopArea:OCE87391003  Train TER
10                  Mantes-la-Jolie       30.0  StopArea:OCE87384008  Train TER


**Contrôles de vraisemblance.** La frontière des 3h30 est cohérente avec les durées commerciales connues
(Montpellier 3h29, Brest 3h26, Quimper 3h28, Clermont-Ferrand 3h16). On vérifie aussi des villes repères,
y compris celles attendues juste au-delà du budget, avant le filtre.

In [ ]:
test = "Lille|Strasbourg|Part.Dieu|Bordeaux|Marseille|Bruxelles|Gen[eè]v|Bâle|Basel|Karlsruhe|Stuttgart"
print(res[res["stop_name"].str.contains(test, case=False)][cols].to_string())

In [ ]:
tout = od.sort_values("duree_min").drop_duplicates("parent_station_d")
tout = tout.merge(stops[["stop_id", "stop_name"]], left_on="parent_station_d", right_on="stop_id")
print(tout[tout["stop_name"].str.contains("Lyon|Marseille|Gen[eè]v|Bâle|Basel", case=False)][cols].to_string())

                   stop_name  duree_min      parent_station_o             produit
164   Lyon Saint-Exupéry TGV      102.0  StopArea:OCE87111849               OUIGO
171           Lyon Part Dieu      105.0  StopArea:OCE87111849           TGV INOUI
208            Lyon Perrache      120.0  StopArea:OCE87111849               OUIGO
311  Marseille Saint-Charles      183.0  StopArea:OCE87686006           TGV INOUI
312                Basel SBB      184.0  StopArea:OCE87686006               Lyria
324          Genève Cornavin      198.0  StopArea:OCE87686006               Lyria
393      Marseille Blancarde      576.0  StopArea:OCE87547000  INTERCITES de nuit


Les gares d'interconnexion (Roissy, Marne-la-Vallée, Massy) donnent parfois le trajet le plus court
(Lyon Part-Dieu en 1h45 depuis Marne-la-Vallée). C'est conforme au cadrage « départ depuis l'Île-de-France » ;
la colonne `parent_station_o` permet de le tracer.

## 1.4 Des gares aux villes

Chaque gare est rattachée à sa commune (ADMIN EXPRESS COG, IGN) par jointure spatiale. Le même geste permet
d'exclure l'Île-de-France (code région 11) et d'agréger les gares par ville. Les gares étrangères, qui ne tombent
dans aucune commune française, reçoivent un nom et une population depuis `data/villes_etrangeres.csv`.

In [ ]:
print(pyogrio.list_layers(ADMIN_EXPRESS))
com = gpd.read_file(ADMIN_EXPRESS, layer="commune")
print(com.crs, len(com))

In [ ]:
pts = res.merge(stops[["stop_id", "stop_lat", "stop_lon"]], on="stop_id")
xy = gpd.points_from_xy(pts["stop_lon"].astype(float), pts["stop_lat"].astype(float))
gares = gpd.GeoDataFrame(pts, geometry=xy, crs="EPSG:4326").to_crs(2154)
com_s = com[["code_insee", "nom_officiel", "population", "code_insee_de_la_region", "geometry"]]
gares = gpd.sjoin(gares, com_s, how="left", predicate="within")
print(len(res), "gares avant /", len(gares), "après jointure")
print(gares["code_insee"].isna().sum(), "gares hors communes françaises :")
print(gares.loc[gares["code_insee"].isna(), "stop_name"].tolist())
print((gares["code_insee_de_la_region"] == "11").sum(), "gares en Île-de-France")

334 gares avant / 334 après jointure
15 gares hors communes françaises :
['Bruxelles Midi', 'Sarrebruck', 'Luxembourg', 'Offenburg', 'Mons', 'Lahr (Schwarzw)', 'Kaiserslautern Hbf', 'Karlsruhe Hbf', 'Ringsheim', 'Mannheim Hbf', 'Freiburg (Breisgau) Hbf', 'Vallorbe', 'Basel SBB', 'Stuttgart Hbf', 'Genève Cornavin']
18 gares en Île-de-France


In [ ]:
etr = pd.read_csv(DATA / "villes_etrangeres.csv", dtype={"stop_name": str, "pays": str}, encoding="utf-8")
g2 = gares[gares["code_insee_de_la_region"] != "11"].merge(etr, on="stop_name", how="left")
g2["cle"] = g2["code_insee"].fillna(g2["pays"] + "_" + g2["ville_etr"])
g2["ville"] = g2["nom_officiel"].fillna(g2["ville_etr"])
g2["population"] = pd.to_numeric(g2["population"].fillna(g2["pop_etr"]))
villes = g2.sort_values("duree_min").drop_duplicates("cle")
print(len(g2), "gares hors IdF /", len(villes), "villes /", villes["population"].isna().sum(), "sans population")
cols = ["cle", "ville", "population", "duree_min", "stop_name", "parent_station_o", "produit", "geometry"]
villes[cols].to_file(GPKG, layer="train_villes")
print(villes.nlargest(10, "population")[["ville", "population", "duree_min", "stop_name"]].to_string())

316 gares hors IdF / 312 villes / 0 sans population
           ville  population  duree_min                  stop_name
293    Marseille    886040.0      183.0    Marseille Saint-Charles
298    Stuttgart    606682.0      189.0              Stuttgart Hbf
153         Lyon    519127.0      105.0             Lyon Part Dieu
185       Nantes    327734.0      117.0                     Nantes
284     Mannheim    320000.0      176.0               Mannheim Hbf
243    Karlsruhe    313000.0      150.0              Karlsruhe Hbf
301  Montpellier    310240.0      192.0  Montpellier Sud de France
152   Strasbourg    293771.0      105.0                 Strasbourg
184     Bordeaux    267991.0      117.0        Bordeaux Saint-Jean
36         Lille    238246.0       55.0               Lille Europe


**Résultat : 312 villes atteignables en train direct**, dont 15 à l'étranger.

# 2. Voiture

L'API publique d'openrouteservice limite les isochrones voiture à 60 minutes. Le contour de 3h30 est donc
reconstruit : on calcule, par l'endpoint *matrix*, le temps de trajet depuis la gare du Nord vers une grille
régulière de points, puis on fusionne les cellules sous chaque seuil.

La clé API est lue dans la variable d'environnement `ORS_API_KEY` (voir le README) et n'est jamais écrite dans le notebook.

## 2.1 Test de l'endpoint matrix

In [ ]:
cle = os.environ["ORS_API_KEY"]
url = "https://api.openrouteservice.org/v2/matrix/driving-car"
pts_test = [ORIG, [3.0573, 50.6365], [4.8357, 45.7640], [4.3517, 50.8503]]  # gare du Nord, Lille, Lyon, Bruxelles
corps = {"locations": pts_test, "sources": [0], "destinations": [1, 2, 3], "metrics": ["duration"]}
r = requests.post(url, json=corps, headers={"Authorization": cle})
r.raise_for_status()
print(r.status_code)
print([round(s / 60) if s else None for s in r.json()["durations"][0]])

Sortie obtenue : `200`, puis `[134, 286, 194]` minutes (Lille, Lyon, Bruxelles). La matrice accepte les trajets longs
et internationaux, sans plafond de portée gênant.

## 2.2 Grille de points

- Projection ETRS89-LAEA (EPSG:3035), métrique et adaptée à l'échelle européenne.
- Disque de 450 km autour de la gare du Nord, pas de 10 km.
- Masque terrestre par les contours de pays GISCO ; Royaume-Uni et îles Anglo-Normandes exclus (pas de traversée maritime).

In [ ]:
orig = gpd.GeoSeries([Point(*ORIG)], crs=4326).to_crs(3035).iloc[0]
R, PAS = 450_000, 10_000
xs = np.arange(orig.x - R, orig.x + R + 1, PAS)
ys = np.arange(orig.y - R, orig.y + R + 1, PAS)
X, Y = np.meshgrid(xs, ys)
grille = gpd.GeoDataFrame(geometry=gpd.points_from_xy(X.ravel(), Y.ravel()), crs=3035)
grille = grille[grille.distance(orig) <= R]
print(len(grille), "points dans le disque")

6361 points dans le disque


In [ ]:
pays = gpd.read_file(PAYS_GISCO)
grille = gpd.sjoin(grille, pays[["CNTR_ID", "geometry"]], predicate="within").drop(columns="index_right")
grille = grille[~grille["CNTR_ID"].isin(["UK", "JE", "GG"])].reset_index(drop=True)
grille["pt_id"] = grille.index
ll = grille.to_crs(4326).geometry
grille["lon"], grille["lat"] = ll.x.round(5), ll.y.round(5)
print(len(grille), "points terrestres retenus")
print(grille["CNTR_ID"].value_counts())

4746 points terrestres retenus
CNTR_ID
FR    3675
DE     438
BE     301
NL     199
CH     106
LU      27
Name: count, dtype: int64


## 2.3 Temps de trajet par lots

In [ ]:
def matrice(ids, profil="driving-car"):
    """Durées (s) depuis ORIG vers les points de la grille désignés par leur pt_id."""
    locs = [ORIG] + grille.loc[ids, ["lon", "lat"]].values.tolist()
    corps = {"locations": locs, "sources": [0], "destinations": list(range(1, len(locs))), "metrics": ["duration"]}
    r = requests.post(f"https://api.openrouteservice.org/v2/matrix/{profil}", json=corps, headers={"Authorization": cle})
    r.raise_for_status()
    return r.json()

Test sur 100 points tirés au hasard : un point non routable renvoie une durée vide sans faire échouer le lot.

In [ ]:
ech = grille.sample(100, random_state=42).index.tolist()
rep = matrice(ech)
d_test = pd.Series(rep["durations"][0], index=ech)
print(d_test.isna().sum(), "points sans durée /", len(d_test))
print((d_test / 60).describe())
print(rep["destinations"][:3])

1 points sans durée / 100
count     99.000000
mean     235.724185
std       77.172948
min       76.469833
25%      167.297667
50%      239.882000
75%      298.902667
max      412.102167
dtype: float64
[{'location': [6.688163, 47.472703], 'snapped_distance': 5.24}, {'location': [5.071838, 49.123603], 'snapped_distance': 239.15}, {'location': [-0.432036, 45.751081], 'snapped_distance': 109.48}]


In [ ]:
morceaux = []
ids = grille.index.tolist()
for i in range(0, len(ids), 1000):
    lot = ids[i:i + 1000]
    morceaux.append(pd.Series(matrice(lot)["durations"][0], index=lot))
    print(i + len(lot), "points traités")
    time.sleep(3)
grille["duree_min"] = pd.concat(morceaux) / 60

1000 points traités
2000 points traités
3000 points traités
4000 points traités
4746 points traités


In [ ]:
print(grille["duree_min"].isna().sum(), "points sans durée")
print(grille["duree_min"].describe())
bord = grille.distance(orig) > R - 20_000
print(((grille["duree_min"] <= BUDGET_MIN) & bord).sum(), "points du bord sous le budget")
grille.to_file(GPKG, layer="grille_voiture")

5 points sans durée
count    4741.000000
mean      227.089768
std        78.840799
min         0.000000
25%       167.541000
50%       233.134500
75%       288.228333
max       462.030333
Name: duree_min, dtype: float64
0 points du bord sous le budget


Aucun point de la couronne extérieure n'est sous le budget : l'isochrone de 3h30 est entièrement contenue
dans le disque, le rayon de 450 km était suffisant.

![Durées voiture sur la grille](../docs/img/voiture_grille_controle.jpg)

## 2.4 Des points aux isochrones

- Les îles d'Yeu et Belle-Île, accessibles uniquement par ferry, sont retirées.
- Les points non routables reçoivent la médiane de leurs voisins (rayon de 15 km).
- Chaque point devient une cellule carrée de 10 km ; les cellules sous chaque seuil sont fusionnées puis découpées par le trait de côte.

In [ ]:
FERRY = [1422, 2050]  # Île d'Yeu et Belle-Île : accès par ferry uniquement
g = grille.drop(index=FERRY).copy()
for pid in g.index[g["duree_min"].isna()]:
    voisins = g.loc[g.distance(g.geometry[pid]) < 15_000, "duree_min"]
    g.loc[pid, "duree_min"] = voisins.median()
    print(pid, "comblé :", round(voisins.median()), "min, d'après", voisins.count(), "voisins")
cell = g.copy()
cell["geometry"] = g.buffer(PAS / 2, cap_style="square")

413 comblé : 345 min, d'après 7 voisins
718 comblé : 350 min, d'après 8 voisins
784 comblé : 374 min, d'après 8 voisins
3566 comblé : 289 min, d'après 8 voisins
4645 comblé : 269 min, d'après 3 voisins


In [ ]:
terre = pays[pays["CNTR_ID"].isin(g["CNTR_ID"].unique())].union_all()
iso = []
for s in SEUILS:
    zone = cell[cell["duree_min"] <= s].union_all().intersection(terre)
    iso.append({"seuil_min": s, "geometry": zone})
iso = gpd.GeoDataFrame(iso, crs=3035)
iso["aire_km2"] = (iso.area / 1e6).round()
print(iso[["seuil_min", "aire_km2"]].to_string())
iso.to_file(GPKG, layer="voiture_isochrones")

Superficies obtenues : 6 600 km² à 1h, 49 991 km² à 2h, 138 474 km² à 3h, **192 809 km² à 3h30**.
La zone s'étire vers le nord-est (Bruxelles, Luxembourg, sud des Pays-Bas), là où le réseau autoroutier est le plus dense.

![Isochrones voiture](../docs/img/voiture_isochrones.jpg)

# 3. Avion

Aucune base ouverte ne donne les horaires de vol. La méthode combine :
1. les **aéroports** d'OurAirports, à moins de 2 500 km de Roissy ou d'Orly ;
2. les **liaisons directes** listées sur les pages Wikipédia des deux aéroports (copie datée sauvegardée dans `data/`) ;
3. un **modèle de durée** linéaire en fonction de la distance orthodromique, calibré sur six vols réels (`data/calibrage_avion.csv`).

## 3.1 Aéroports et calibrage du modèle

In [ ]:
geod = Geod(ellps="WGS84")
apt = pd.read_csv(AIRPORTS, dtype=str, keep_default_na=False)  # « NA » = Namibie, à ne pas lire comme une valeur manquante
apt = apt[(apt["iata_code"] != "") & (apt["scheduled_service"] == "yes")].drop_duplicates("iata_code").set_index("iata_code")
print(len(apt), "aéroports avec code IATA et service régulier")

4133 aéroports avec code IATA et service régulier


In [ ]:
calib = pd.read_csv(DATA / "calibrage_avion.csv")

def dist_km(o, d):
    a, b = apt.loc[o], apt.loc[d]
    return geod.inv(float(a["longitude_deg"]), float(a["latitude_deg"]), float(b["longitude_deg"]), float(b["latitude_deg"]))[2] / 1000

calib["dist_km"] = [dist_km(o, d) for o, d in zip(calib["origine"], calib["destination_iata"])]
pente, fixe = np.polyfit(calib["dist_km"], calib["duree_min"], 1)
calib["modele"] = fixe + pente * calib["dist_km"]
calib["residu"] = calib["duree_min"] - calib["modele"]
print(f"durée = {fixe:.0f} min + distance / {60 / pente:.0f} km/h")
print(calib[["origine", "destination_iata", "dist_km", "duree_min", "modele", "residu"]].round(0).to_string())

durée = 38 min + distance / 758 km/h
  origine destination_iata  dist_km  duree_min  modele  residu
0     CDG              SEN    314.0         65    63.0     2.0
1     ORY              BCN    826.0        100   103.0    -3.0
2     CDG              FCO   1102.0        120   125.0    -5.0
3     ORY              LIS   1438.0        160   152.0     8.0
4     CDG              ATH   2111.0        190   205.0   -15.0
5     CDG              IST   2217.0        225   213.0    12.0


Le modèle (**38 min + distance à 758 km/h**) est physiquement plausible. Les résidus atteignent ±15 minutes aux
longues distances : les destinations modélisées entre 195 et 225 minutes sont signalées comme incertaines.

In [ ]:
for code_apt in ["CDG", "ORY"]:
    o_apt = apt.loc[code_apt]
    n = len(apt)
    apt["d_" + code_apt] = geod.inv(np.full(n, float(o_apt["longitude_deg"])), np.full(n, float(o_apt["latitude_deg"])),
                                    apt["longitude_deg"].astype(float).values, apt["latitude_deg"].astype(float).values)[2] / 1000
cand = apt[apt["type"].isin(["large_airport", "medium_airport"]) & (apt[["d_CDG", "d_ORY"]].min(axis=1) <= 2500)]
cand = cand.drop(index=["CDG", "ORY"], errors="ignore").copy()
print(len(cand), "aéroports candidats")
print(cand["iso_country"].value_counts().head(15))

514 aéroports candidats
iso_country
FR    44
IT    41
GB    40
ES    35
SE    34
DE    29
NO    28
DZ    25
GR    24
FI    19
MA    16
RO    16
PL    15
TR    12
DK    10
Name: count, dtype: int64


## 3.2 Liaisons directes depuis Roissy et Orly

Les pages Wikipédia (en anglais) sont téléchargées une seule fois, puis relues depuis la copie locale :
c'est cette copie datée qui fait référence. Les vols charters sont exclus, les vols saisonniers conservés,
et les lignes ouvertes après ou fermées avant le jour de référence sont retirées.

In [ ]:
pages = {"CDG": "Charles_de_Gaulle_Airport", "ORY": "Orly_Airport"}
dest = []
for code_apt, page in pages.items():
    copie = DATA / f"wiki_{code_apt}.html"
    if not copie.exists():
        html = requests.get(f"https://en.wikipedia.org/wiki/{page}", headers={"User-Agent": "zones3h-portfolio/1.0"}).text
        copie.write_text(html, encoding="utf-8")
    html = re.sub(r"<br\s*/?>", ", ", copie.read_text(encoding="utf-8"))
    tab = [t for t in pd.read_html(StringIO(html)) if "Destinations" in t.columns][0]
    for cellule in tab["Destinations"].astype(str):
        cellule = re.split(r"Seasonal charter:|Charter:", re.sub(r"\[\d+\]", "", cellule))[0].replace("Seasonal:", ",")
        dest += [(code_apt, x.strip()) for x in cellule.split(",") if x.strip()]
dest = pd.DataFrame(dest, columns=["origine", "dest_wiki"]).drop_duplicates()

In [ ]:
ann = dest["dest_wiki"].str.extract(r"\((begins|ends) (\d+ \w+ \d{4})\)")
date_ann = pd.to_datetime(ann[1], format="%d %B %Y")
ref = pd.Timestamp(JOUR)
hors = ((ann[0] == "begins") & (date_ann > ref)) | ((ann[0] == "ends") & (date_ann < ref))
dest = dest[~hors].copy()
dest["dest_wiki"] = dest["dest_wiki"].str.replace(r"\s*\((begins|ends)[^)]*\)", "", regex=True).str.strip()
dest = dest.drop_duplicates()
print(len(dest), "couples après traitement des dates")

458 couples après traitement des dates


Rapprochement des noms Wikipédia avec OurAirports : recherche par mot entier (accents et casse neutralisés)
dans le nom de l'aéroport et dans sa commune, en privilégiant l'indice d'aéroport (« Paris–Orly ») puis la ville.

In [ ]:
def norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()

cand["ville_n"], cand["nom_n"] = cand["municipality"].map(norm), cand["name"].map(norm)

def trouver(ville, indice):
    for terme in [indice, ville.split("/")[0]]:
        if not terme:
            continue
        motif = r"\b" + re.escape(norm(terme)) + r"\b"
        c = cand[cand["nom_n"].str.contains(motif) | cand["ville_n"].str.contains(motif)]
        if len(c):
            return c.sort_values("type").index[0]
    return None

dest[["ville", "indice"]] = dest["dest_wiki"].str.split("–", n=1, expand=True)
dest["iata"] = [trouver(v, i) for v, i in zip(dest["ville"], dest["indice"])]
print(dest["iata"].notna().sum(), "rapprochées /", len(dest))

Les destinations non rapprochées sont, pour l'essentiel, hors du rayon de 2 500 km (long-courrier).
Les cas restants (noms fusionnés, orthographes différentes) et Istanbul (rattaché à tort à Sabiha Gökçen)
sont corrigés par une table de correspondance explicite.

In [ ]:
MANUEL = {"Ajaccio Bari": ["AJA", "BRI"], "Fès Oujda": ["FEZ", "OUD"], "Gothenburg": ["GOT"], "Kefalonia": ["EFL"],
          "Tlemcen": ["TLM"], "Izmir": ["ADB"], "İzmir": ["ADB"], "Basel/Mulhouse": ["BSL"], "Ancona": ["AOI"],
          "Le Puy": ["LPY"], "Istanbul": ["IST"]}
dest.loc[dest["dest_wiki"].isin(MANUEL.keys()), "iata"] = None
ajout = [(o, c) for o, w in zip(dest["origine"], dest["dest_wiki"]) if w in MANUEL for c in MANUEL[w]]
vols = pd.concat([dest.dropna(subset=["iata"])[["origine", "iata"]], pd.DataFrame(ajout, columns=["origine", "iata"])])
vols = vols.drop_duplicates().reset_index(drop=True)
apt_all = pd.read_csv(AIRPORTS, dtype=str, keep_default_na=False)
apt_all = apt_all[apt_all["iata_code"] != ""].drop_duplicates("iata_code").set_index("iata_code")
print(len(vols), "liaisons directes /", vols["iata"].nunique(), "aéroports de destination")
print("codes introuvables :", [c for c in vols["iata"].unique() if c not in apt_all.index])

275 liaisons directes / 184 aéroports de destination
codes introuvables : []


## 3.3 Durées et couche finale

In [ ]:
def coord(c):
    return float(apt_all.at[c, "longitude_deg"]), float(apt_all.at[c, "latitude_deg"])

vols["dist_km"] = [geod.inv(*coord(o), *coord(d))[2] / 1000 for o, d in zip(vols["origine"], vols["iata"])]
vols["duree_mod"] = fixe + pente * vols["dist_km"]
av = vols.sort_values("duree_mod").drop_duplicates("iata").copy()
av["ville"], av["pays"] = av["iata"].map(apt_all["municipality"]), av["iata"].map(apt_all["iso_country"])
bande = av[av["duree_mod"].between(195, 225)]
print((av["duree_mod"] <= BUDGET_MIN).sum(), "destinations sous 3h30 selon le modèle /", len(bande), "dans la bande d'incertitude")
print(bande[["origine", "iata", "ville", "pays", "dist_km", "duree_mod"]].round(0).to_string())

165 destinations sous 3h30 selon le modèle / 21 dans la bande
    origine iata             ville pays  dist_km  duree_mod
224     ORY  RMO          Chişinău   MD   1989.0      195.0
242     ORY  JSI          Skiathos   GR   1992.0      195.0
115     CDG  VAR             Varna   BG   2045.0      200.0
230     ORY  KLX          Kalamata   GR   2053.0      200.0
222     ORY  BOJ            Burgas   BG   2066.0      201.0
183     ORY  RAK         Marrakesh   MA   2093.0      203.0
165     ORY  ATH    Spata-Artemida   GR   2110.0      205.0
204     ORY  OZZ        Ouarzazate   MA   2125.0      206.0
196     ORY  ESU         Essaouira   MA   2175.0      210.0
234     ORY  LLA             Luleå   SE   2201.0      212.0
32      CDG  AEY          Akureyri   IS   2209.0      213.0
266     CDG  IST          Istanbul   TR   2217.0      213.0
236     ORY  JMK           Mykonos   GR   2240.0      215.0
124     CDG  KEF         Reykjavík   IS   2247.0      216.0
180     ORY  AGA   Agadir (Temsia)   M

In [ ]:
REEL = {"ATH": 190, "IST": 225}  # durées relevées lors du calibrage (Google Flights, 08/10/2026)
av["duree_min"] = av["iata"].map(REEL).fillna(av["duree_mod"])
av["statut"] = np.where(av["iata"].isin(REEL), "relevé", "modélisé")
av["limite"] = av["duree_min"].between(195, 225) & (av["statut"] == "modélisé")
av = av[av["duree_min"] <= BUDGET_MIN].copy()
xy = gpd.points_from_xy([coord(c)[0] for c in av["iata"]], [coord(c)[1] for c in av["iata"]])
avion = gpd.GeoDataFrame(av, geometry=xy, crs=4326).to_crs(3035)
print(len(avion), "destinations /", avion["limite"].sum(), "à la limite")
print(avion["pays"].value_counts().head(10))
avion.to_file(GPKG, layer="avion_villes")

165 destinations / 8 à la limite
pays
FR    26
IT    21
ES    15
GB    14
DZ    10
MA    10
GR     8
DE     7
TN     5
IE     3
Name: count, dtype: int64


**Résultat : 165 destinations**, dont 8 proches du seuil (champ `limite`). Contrôle des liaisons d'hiver
(stations de montagne) : seule Turin, desservie toute l'année, figure dans la sélection.

![Destinations aériennes](../docs/img/avion_destinations.jpg)

# 4. Marche et vélo

Pour ces profils, l'API d'isochrones openrouteservice accepte des durées de plusieurs heures (20 h à pied, 5 h à vélo) :
les quatre seuils sont obtenus en une requête par mode.

In [ ]:
for profil, nom in [("foot-walking", "marche"), ("cycling-regular", "velo")]:
    corps = {"locations": [ORIG], "range": [s * 60 for s in SEUILS], "range_type": "time"}
    r = requests.post(f"https://api.openrouteservice.org/v2/isochrones/{profil}", json=corps, headers={"Authorization": cle})
    r.raise_for_status()
    gdf = gpd.GeoDataFrame.from_features(r.json()["features"], crs=4326).to_crs(3035)
    gdf["seuil_min"] = (gdf["value"] / 60).round().astype(int)
    gdf["aire_km2"] = (gdf.area / 1e6).round()
    gdf[["seuil_min", "aire_km2", "geometry"]].to_file(GPKG, layer=f"{nom}_isochrones")
    print(nom, "\n", gdf[["seuil_min", "aire_km2"]].to_string())

marche 
    seuil_min  aire_km2
0         60      67.0
1        120     272.0
2        180     608.0
3        210     822.0
velo 
    seuil_min  aire_km2
0         60     786.0
1        120    3059.0
2        180    6927.0
3        210    9505.0


| Mode | 1h | 3h30 | Rapport | Lecture |
|---|---|---|---|---|
| Marche | 67 km² | 822 km² | 12,3 | croissance régulière (vitesse constante : 3,5² ≈ 12,25) |
| Vélo | 786 km² | 9 505 km² | 12,1 | croissance régulière |
| Voiture | 6 600 km² | 192 809 km² | 29,2 | croissance accélérée par le réseau autoroutier |

| Marche | Vélo |
|---|---|
| ![Isochrones à pied](../docs/img/marche_isochrones.jpg) | ![Isochrones à vélo](../docs/img/velo_isochrones.jpg) |

# 5. Synthèse

Point de reprise : les couches sont rechargées depuis le GeoPackage, ce qui permet d'exécuter cette partie seule.

In [ ]:
villes = gpd.read_file(GPKG, layer="train_villes").to_crs(3035)
grille_v = gpd.read_file(GPKG, layer="grille_voiture")
avion = gpd.read_file(GPKG, layer="avion_villes")
com = gpd.read_file(ADMIN_EXPRESS, layer="commune")
print(villes.crs.to_epsg(), grille_v.crs.to_epsg(), avion.crs.to_epsg(), "|", len(villes), len(grille_v), len(avion), len(com))

3035 3035 3035 | 312 4746 165 34746


## 5.1 Comparaison des modes pour les villes du train

Durée voiture de la cellule la plus proche (7,1 km au plus), durée avion de l'aéroport le plus proche (30 km au plus).

In [ ]:
v = villes[["cle", "ville", "population", "duree_min", "geometry"]].rename(columns={"duree_min": "train_min"})
v = gpd.sjoin_nearest(v, grille_v[["duree_min", "geometry"]], how="left", max_distance=7_100).rename(columns={"duree_min": "voiture_min"}).drop(columns="index_right")
v = gpd.sjoin_nearest(v, avion[["iata", "duree_min", "geometry"]], how="left", max_distance=30_000).rename(columns={"duree_min": "avion_min"}).drop(columns="index_right")
v = v.drop_duplicates("cle")
v["plus_rapide"] = v[["train_min", "voiture_min", "avion_min"]].idxmin(axis=1).str.replace("_min", "")
print(v["plus_rapide"].value_counts())
print(v.nlargest(10, "population")[["ville", "train_min", "voiture_min", "avion_min", "plus_rapide"]].round(0).to_string())
v.to_file(GPKG, layer="comparaison_villes")

plus_rapide
train      283
avion       19
voiture     10
Name: count, dtype: int64
           ville  train_min  voiture_min  avion_min plus_rapide
290    Marseille      183.0          NaN       87.0       avion
295    Stuttgart      189.0          NaN        NaN       train
152         Lyon      105.0        284.0       70.0       avion
184       Nantes      117.0        241.0       67.0       avion
281     Mannheim      176.0          NaN        NaN       train
240    Karlsruhe      150.0        323.0        NaN       train
298  Montpellier      192.0          NaN       84.0       avion
151   Strasbourg      105.0        290.0        NaN       train
183     Bordeaux      117.0          NaN       79.0       avion
36         Lille       55.0        136.0        NaN       train


L'avion « gagne » pour Marseille, Lyon, Nantes ou Bordeaux : c'est l'effet direct du cadrage en **temps à bord**,
qui ignore l'accès à l'aéroport et les formalités. En temps porte-à-porte, le classement s'inverserait.

## 5.2 Communes atteintes à vélo hors Île-de-France

In [ ]:
velo = gpd.read_file(GPKG, layer="velo_isochrones")
zone = velo.loc[velo["seuil_min"] == BUDGET_MIN, "geometry"].iloc[0]
pts_com = com[com["code_insee_de_la_region"] != "11"].copy()
pts_com["geometry"] = pts_com.representative_point()
pts_com = pts_com.to_crs(3035)
dedans = pts_com[pts_com.within(zone)].sort_values("population", ascending=False)
print(len(dedans), "communes hors Île-de-France atteintes à vélo en 3h30")
print(dedans[["nom_officiel", "code_insee_du_departement", "population"]].head(10).to_string())

191 communes hors Île-de-France atteintes à vélo en 3h30
              nom_officiel code_insee_du_departement  population
22564                Creil                        60       36301
22842      Nogent-sur-Oise                        60       21907
22985               Senlis                        60       15157
22797           Montataire                        60       14257
22778                 Méru                        60       13897
22886  Pont-Sainte-Maxence                        60       12361
22530            Chantilly                        60       10591
22546             Clermont                        60       10564
22528              Chambly                        60        9959
22731            Lamorlaye                        60        9082


In [ ]:
print(dedans["code_insee_du_departement"].value_counts())

code_insee_du_departement
60    191
Name: count, dtype: int64


# 6. Préparation des couches pour la cartographie

- Les isochrones emboîtées sont converties en **anneaux** (0–60, 60–120, 120–180, 180–210 min) pour éviter la superposition des aplats.
- Les points du train et de l'avion reçoivent une classe de durée, et le train une classe de taille selon la population.
- Les couches de la storymap sont exportées en WGS84 (GeoJSON et GeoPackage) pour ArcGIS Online.

In [ ]:
for nom in ["marche", "velo", "voiture"]:
    iso_m = gpd.read_file(GPKG, layer=f"{nom}_isochrones").sort_values("seuil_min").reset_index(drop=True)
    geoms = [iso_m.geometry[0]] + [iso_m.geometry[i].difference(iso_m.geometry[i - 1]) for i in range(1, len(iso_m))]
    bornes = [0] + iso_m["seuil_min"].tolist()
    anneaux = gpd.GeoDataFrame(iso_m.drop(columns="geometry"), geometry=geoms, crs=iso_m.crs)
    anneaux["classe"] = [f"{bornes[i]}–{bornes[i + 1]} min" for i in range(len(iso_m))]
    anneaux.to_file(GPKG, layer=f"{nom}_anneaux")
    print(nom, anneaux["classe"].tolist())

marche ['0–60 min', '60–120 min', '120–180 min', '180–210 min']
velo ['0–60 min', '60–120 min', '120–180 min', '180–210 min']
voiture ['0–60 min', '60–120 min', '120–180 min', '180–210 min']


In [ ]:
gpd.GeoDataFrame({"nom": ["Départ : gare du Nord"]}, geometry=[Point(*ORIG)], crs=4326).to_file(GPKG, layer="depart")

bornes, libelles = [0, 60, 120, 180, 210], ["0–60 min", "60–120 min", "120–180 min", "180–210 min"]
for couche in ["train_villes", "avion_villes"]:
    gdf = gpd.read_file(GPKG, layer=couche)
    gdf["classe"] = pd.cut(gdf["duree_min"], bins=bornes, labels=libelles, include_lowest=True).astype(str)
    if "population" in gdf:
        gdf["taille"] = pd.cut(gdf["population"], bins=[0, 20_000, 100_000, float("inf")], labels=["petite", "moyenne", "grande"]).astype(str)
    gdf.to_file(GPKG, layer=couche)
    print(couche, gdf["classe"].value_counts().sort_index().to_dict())

train_villes {'0–60 min': 53, '120–180 min': 96, '180–210 min': 25, '60–120 min': 138}
avion_villes {'0–60 min': 3, '120–180 min': 64, '180–210 min': 18, '60–120 min': 80}


In [ ]:
couches = ["marche_anneaux", "velo_anneaux", "voiture_anneaux", "train_villes", "avion_villes", "depart"]
(OUT / "geojson").mkdir(exist_ok=True)
for c in couches:
    gdf = gpd.read_file(GPKG, layer=c).to_crs(4326)
    gdf.to_file(OUT / "geojson" / f"{c}.geojson", driver="GeoJSON")
    gdf.to_file(OUT / "storymap.gpkg", layer=c)
    print(c, len(gdf), "entités,", round(os.path.getsize(OUT / "geojson" / f"{c}.geojson") / 1024), "Ko")

marche_anneaux 4 entités, 34 Ko
velo_anneaux 4 entités, 165 Ko
voiture_anneaux 4 entités, 53 Ko
train_villes 312 entités, 93 Ko
avion_villes 165 entités, 51 Ko
depart 1 entités, 0 Ko
